In [ ]:
import json
from datetime import datetime
from collections import defaultdict
import numpy as np

with open("./ips_continuous_results.json", "r") as f:
    data = json.load(f)

# Filter only real sources
source_entries = [d for d in data if d.get("type") == "source"]

# Group by source name
grouped = defaultdict(list)

for entry in source_entries:
    t = datetime.fromisoformat(entry["start_time"])
    grouped[entry["source"]].append(t)

avg_gaps = {}

for src, times in grouped.items():
    if len(times) < 2:
        continue  # cannot compute gap
    
    times_sorted = sorted(times)
    gaps = [
        (times_sorted[i+1] - times_sorted[i]).total_seconds()
        for i in range(len(times_sorted)-1)
    ]
    
    avg_gaps[src] = np.mean(gaps)

# Overall average revisit time across all sources
overall_avg = np.mean(list(avg_gaps.values()))

print("Average revisit time per source (seconds):")
for src, val in avg_gaps.items():
    print(f"{src}: {val:.2f} s")

print("\nOverall average revisit time:", overall_avg/86400, "days")

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde

# ── Matplotlib style configuration ───────────────────────────────────────────
plt.rcParams.update({
    # Font
    "font.family":        "serif",
    "font.serif":         ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset":   "stix",

    # Axes
    "axes.linewidth":     0.8,
    "axes.labelsize":     11,
    "axes.titlesize":     12,
    "axes.titlepad":      10,
    "axes.spines.top":    True,
    "axes.spines.right":  True,

    # Ticks
    "xtick.direction":    "in",
    "ytick.direction":    "in",
    "xtick.major.size":   4,
    "ytick.major.size":   4,
    "xtick.minor.size":   2,
    "ytick.minor.size":   2,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "xtick.top":          True,
    "ytick.right":        True,

    # Legend
    "legend.fontsize":    9,
    "legend.framealpha":  0.9,
    "legend.edgecolor":   "0.7",

    # Figure
    "figure.dpi":         150,
    "savefig.dpi":        150,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
})

# ── Load data ─────────────────────────────────────────────────────────────────
with open("./ips_continuous_results.json", "r") as f:
    data = json.load(f)

In [ ]:
ra  = np.array([d["ra"]  for d in data if d["type"] == "source"])
dec = np.array([d["dec"] for d in data if d["type"] == "source"])

# ── Compute point density for colour mapping ──────────────────────────────────
xy  = np.vstack([ra, dec])
kde = gaussian_kde(xy, bw_method=0.15)
z   = kde(xy)
order = z.argsort()          # draw dense regions on top
ra_s, dec_s, z_s = ra[order], dec[order], z[order]

# ── Custom colourmap (white → steel-blue → navy) ──────────────────────────────
cmap = LinearSegmentedColormap.from_list(
    "ips",
    ["#d0e4f7", "#4a90c4", "#1a3a5c"],
    N=256,
)

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(7.0, 4.8))

# Main scatter axis (left ~80 %)
ax  = fig.add_axes([0.10, 0.12, 0.72, 0.78])

# Colourbar axis
cax = fig.add_axes([0.84, 0.12, 0.025, 0.78])

# ── Scatter plot ──────────────────────────────────────────────────────────────
sc = ax.scatter(
    ra_s, dec_s,
    c=z_s,
    cmap=cmap,
    s=4,
    linewidths=0,
    alpha=0.85,
    rasterized=True,      # keeps vector PDF small
)

# ── Colourbar ─────────────────────────────────────────────────────────────────
cb = fig.colorbar(sc, cax=cax)
cb.set_label("Relative source density", labelpad=8, fontsize=9)
cb.ax.tick_params(labelsize=8, direction="in")
cb.outline.set_linewidth(0.6)

# ── Source count annotation ───────────────────────────────────────────────────
ax.text(
    0.97, 0.97,
    rf"$N = {len(ra):,}$",
    transform=ax.transAxes,
    ha="right", va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.75", lw=0.6),
)

# ── Labels & title ────────────────────────────────────────────────────────────
ax.set_xlabel(r"Right Ascension $\alpha$ (deg)")
ax.set_ylabel(r"Declination $\delta$ (deg)")
ax.set_title("IPS Source Distribution", fontweight="normal")

# ── Tick formatting ───────────────────────────────────────────────────────────
ax.xaxis.set_major_locator(ticker.AutoLocator())
ax.yaxis.set_major_locator(ticker.AutoLocator())
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

# ── Invert RA axis (astronomical convention) ──────────────────────────────────
if ax.get_xlim()[0] < ax.get_xlim()[1]:
        ax.invert_xaxis()

# ── Save & show ───────────────────────────────────────────────────────────────
fig.savefig("ips_source_distribution.png", dpi=300, bbox_inches="tight")   # raster preview
plt.show()
plt.close()

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde

# ── Matplotlib style configuration ───────────────────────────────────────────
plt.rcParams.update({
    # Font
    "font.family":         "serif",
    "font.serif":          ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset":    "stix",

    # Axes
    "axes.linewidth":      0.8,
    "axes.labelsize":      11,
    "axes.titlesize":      12,
    "axes.titlepad":       10,
    "axes.spines.top":     True,
    "axes.spines.right":   True,

    # Ticks
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.major.size":    4,
    "ytick.major.size":    4,
    "xtick.minor.size":    2,
    "ytick.minor.size":    2,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.labelsize":     9,
    "ytick.labelsize":     9,
    "xtick.top":           True,
    "ytick.right":         True,

    # Legend
    "legend.fontsize":     9,
    "legend.framealpha":   0.9,
    "legend.edgecolor":    "0.7",

    # Figure
    "figure.dpi":          150,
    "savefig.dpi":         300,
    "savefig.bbox":        "tight",
    "savefig.pad_inches":  0.05,
})

# ── Load data ─────────────────────────────────────────────────────────────────
with open("./ips_continuous_results.json", "r") as f:
    data = json.load(f)

src    = [d for d in data if d["type"] == "source"]
elong  = np.array([d["elong_deg"] for d in src])
si     = np.array([d["si"]        for d in src])

# ── Filter out non-positive SI (required for log scale & KDE) ────────────────
mask  = (si > 0) & np.isfinite(si) & np.isfinite(elong)
elong = elong[mask]
si    = si[mask]

# ── KDE density in log-SI space (gives better colour spread on log axis) ─────
log_si = np.log10(si)
xy     = np.vstack([elong, log_si])
kde    = gaussian_kde(xy, bw_method=0.15)
z      = kde(xy)
order  = z.argsort()                         # dense regions drawn on top
e_s, si_s, z_s = elong[order], si[order], z[order]

# ── Custom colourmap — warm (sparse = pale gold → dense = deep crimson) ───────
cmap = LinearSegmentedColormap.from_list(
    "si_elong",
    ["#fef3c7", "#f97316", "#7f1d1d"],
    N=256,
)

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(7.0, 4.8))
ax  = fig.add_axes([0.10, 0.12, 0.72, 0.78])
cax = fig.add_axes([0.84, 0.12, 0.025, 0.78])

# ── Scatter ───────────────────────────────────────────────────────────────────
sc = ax.scatter(
    e_s, si_s,
    c=z_s,
    cmap=cmap,
    s=4,
    linewidths=0,
    alpha=0.80,
    rasterized=True,
)

# ── Log scale & ticks ─────────────────────────────────────────────────────────
ax.set_yscale("log")
ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation(base=10))
ax.yaxis.set_minor_locator(ticker.LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=20))
ax.xaxis.set_major_locator(ticker.AutoLocator())
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())

# ── Colourbar ─────────────────────────────────────────────────────────────────
cb = fig.colorbar(sc, cax=cax)
cb.set_label("Relative source density", labelpad=8, fontsize=9)
cb.ax.tick_params(labelsize=8, direction="in")
cb.outline.set_linewidth(0.6)

# ── Source count annotation ───────────────────────────────────────────────────
ax.text(
    0.03, 0.04,
    rf"$N = {len(ra):,}$",
    transform=ax.transAxes,
    ha="left", va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.75", lw=0.6),
)

# ── Labels & title ────────────────────────────────────────────────────────────
ax.set_xlabel(r"Solar Elongation $\varepsilon$ (deg)")
ax.set_ylabel(r"Scintillation Index $m$")
ax.set_title("Scintillation Index vs. Solar Elongation",
             fontweight="normal")

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig("ips_si_vs_elongation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from datetime import datetime

# ── Matplotlib style configuration ───────────────────────────────────────────
plt.rcParams.update({
    # Font
    "font.family":         "serif",
    "font.serif":          ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset":    "stix",

    # Axes
    "axes.linewidth":      0.8,
    "axes.labelsize":      20,
    "axes.titlesize":      20,
    "axes.titlepad":       20,
    "axes.spines.top":     True,
    "axes.spines.right":   True,

    # Ticks
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.major.size":    4,
    "ytick.major.size":    4,
    "xtick.minor.size":    2,
    "ytick.minor.size":    2,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.labelsize":     9,
    "ytick.labelsize":     9,
    "xtick.top":           True,
    "ytick.right":         True,

    # Legend
    "legend.fontsize":     9,
    "legend.framealpha":   0.9,
    "legend.edgecolor":    "0.7",

    # Figure
    "figure.dpi":          300,
    "savefig.dpi":         300,
    "savefig.bbox":        "tight",
    "savefig.pad_inches":  0.05,
})



# ── Custom colourmap: light lavender → blue → deep indigo ────────────────────
cmap = LinearSegmentedColormap.from_list(
    "si_cool",
    ["#e8eaf6", "#7986cb", "#1a237e", "#0d0d3b"],
    N=256,
)

# ── Load data ─────────────────────────────────────────────────────────────────
with open("./ips_continuous_results.json", "r") as f:
    data = json.load(f)

src_data = []
for d in data:
    if d.get("type") == "source":
        try:
            dt = datetime.fromisoformat(d["start_time"])
            src_data.append({
                "ra":    d["ra"],
                "dec":   d["dec"],
                "si":    d["si"],
                "month": dt.strftime("%Y-%m")
            })
        except KeyError:
            continue

months = sorted(set(d["month"] for d in src_data))[:8]  # cap at 8

# ── Layout: 2 rows × 4 cols ───────────────────────────────────────────────────
ncols, nrows = 4, 2
fig = plt.figure(figsize=(22, 12))
fig.patch.set_facecolor("white")

axes = []
for i in range(nrows * ncols):
    ax = fig.add_subplot(nrows, ncols, i + 1, projection="3d")
    axes.append(ax)

# Shared normalisation across all months
all_log_si = []
for m in months:
    filtered = [d for d in src_data if d["month"] == m]
    si = np.array([d["si"] for d in filtered])
    mask = (si > 0) & np.isfinite(si)
    si = si[mask]
    if len(si):
        all_log_si.append(np.log10(si))
if all_log_si:
    combined = np.concatenate(all_log_si)
    vmin, vmax = np.percentile(combined, 2), np.percentile(combined, 98)
else:
    vmin, vmax = -1, 1

# ── Draw each month ───────────────────────────────────────────────────────────
sc_ref = None  # for the colorbar

for idx, (month, ax) in enumerate(zip(months, axes)):
    filtered = [d for d in src_data if d["month"] == month]
    if not filtered:
        ax.set_visible(False)
        continue

    ra  = np.array([d["ra"]  for d in filtered])
    dec = np.array([d["dec"] for d in filtered])
    si  = np.array([d["si"]  for d in filtered])

    mask = (si > 0) & np.isfinite(si) & np.isfinite(ra) & np.isfinite(dec)
    ra, dec, si = ra[mask], dec[mask], si[mask]
    if len(si) == 0:
        ax.set_visible(False)
        continue

    log_si = np.log10(si)

    # RA/Dec → Cartesian
    ra_rad  = np.radians(ra)
    dec_rad = np.radians(dec)
    x = np.cos(dec_rad) * np.cos(ra_rad)
    y = np.cos(dec_rad) * np.sin(ra_rad)
    z = np.sin(dec_rad)

    order = log_si.argsort()
    x, y, z, log_si = x[order], y[order], z[order], log_si[order]

    # Background wireframe sphere
    u = np.linspace(0, 2 * np.pi, 40)
    v = np.linspace(0, np.pi, 20)
    sx = np.outer(np.cos(u), np.sin(v))
    sy = np.outer(np.sin(u), np.sin(v))
    sz = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(sx, sy, sz, color="#ccccdd", alpha=0.5, linewidth=0.4)

    # Scatter
    sc = ax.scatter(
        x, y, z,
        c=log_si,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        s=6,
        linewidths=0,
        alpha=0.9,
        rasterized=True,
    )
    sc_ref = sc

    # Style axes
    ax.set_box_aspect([1, 1, 1])
    ax.set_axis_off()
    ax.set_facecolor("white")
    # Shrink axis limits so the unit sphere fills more of the panel
    ax.set_xlim(-0.70, 0.70)
    ax.set_ylim(-0.70, 0.70)
    ax.set_zlim(-0.70, 0.70)

    # Month label
    label = datetime.strptime(month, "%Y-%m").strftime("%b %Y")
    ax.set_title(
        f"{label}\n$N={len(x):,}$",
        color="#222222",
        # fontsize=8,
        pad=2,
        fontweight="normal",
    )

# Hide any unused axes
for idx in range(len(months), nrows * ncols):
    axes[idx].set_visible(False)

# ── Shared colourbar ──────────────────────────────────────────────────────────
if sc_ref is not None:
    cax = fig.add_axes([0.92, 0.15, 0.012, 0.70])
    cb  = fig.colorbar(sc_ref, cax=cax)
    cb.set_label(
        r"Scintillation Index $\log_{10}(m)$",
        color="#222222",
        labelpad=20,
        fontsize=20,
    )
    cb.ax.tick_params(labelsize=11, colors="#222222", direction="in")
    cb.outline.set_edgecolor("#888888")
    cb.outline.set_linewidth(0.5)
    plt.setp(cb.ax.yaxis.get_ticklabels(), color="#222222")

# ── Main title ────────────────────────────────────────────────────────────────
fig.suptitle(
    "Monthly Scintillation Index Measurement Plots",
    color="#000000",
    fontsize=25,
    y=1.05,
    fontweight="normal",
)

fig.subplots_adjust(
    left=0.01, right=0.90,
    top=0.94, bottom=0.02,
    wspace=0.02, hspace=0.12,
)

out = "ips_celestial_8months.png"
fig.savefig(out, dpi=150, bbox_inches="tight",
            facecolor="white", transparent=False)
# plt.close(fig)
print(f"Saved: {out}")
plt.show()

In [ ]:
import json

with open("./ips_continuous_results.json") as f:
    data = json.load(f)

sources = [d for d in data if d["type"] == "source"]

unique = {}
for s in sources:
    if s["source"] not in unique:
        unique[s["source"]] = (s["ra"], s["dec"])

items = sorted(unique.items(), key=lambda x: x[1][0])

# group into pairs
rows = [items[i:i+2] for i in range(0, len(items), 2)]

with open("sources.tex", "w") as f:
    f.write("\\setlength{\\tabcolsep}{4pt}\n")
    f.write("\\renewcommand{\\arraystretch}{1.1}\n")

    f.write("\\begin{longtable}{lcc lcc}\n")
    f.write("\\caption{Unique IPS sources used in this work} \\\\\n")

    f.write("\\toprule\n")
    f.write("Source & RA (deg) & Dec (deg) & Source & RA (deg) & Dec (deg) \\\\\n")
    f.write("\\midrule\n")
    f.write("\\endfirsthead\n")

    f.write("\\toprule\n")
    f.write("Source & RA (deg) & Dec (deg) & Source & RA (deg) & Dec (deg) \\\\\n")
    f.write("\\midrule\n")
    f.write("\\endhead\n")

    f.write("\\midrule\n")
    f.write("\\multicolumn{6}{r}{Continued on next page} \\\\\n")
    f.write("\\midrule\n")
    f.write("\\endfoot\n")

    f.write("\\bottomrule\n")
    f.write("\\endlastfoot\n")

    for row in rows:
        line = ""
        for name, (ra, dec) in row:
            line += f"{name} & {ra:.4f} & {dec:.4f} & "

        line = line.rstrip("& ")

        if len(row) == 1:
            line += " & & &"

        f.write(line + " \\\\\n")

    f.write("\\end{longtable}\n")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.linewidth": 0.8,
    "axes.labelsize": 11,
    "axes.titlesize": 15,
    "axes.titlepad": 10,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "xtick.top": True,
    "ytick.right": True,
    "legend.fontsize": 9,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "0.7",
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})

# =========================
# CME WINDOWS
# =========================
CME_WINDOWS = [
    ("2011-06-06", "2011-06-07", "CME-1\n(Jun 6–7)",  "#d62728"),
    ("2011-06-18", "2011-06-21", "CME-2\n(Jun 18–21)", "#1f77b4"),
]

# =========================
# LOAD + CLEAN
# =========================
df = pd.read_json("ips_continuous_results.json")
df = df[df["type"] == "source"].copy()
df["start_time"] = pd.to_datetime(df["start_time"])
df = df.sort_values("start_time")

june = df[
    (df["start_time"] >= "2011-06-01") &
    (df["start_time"] <= "2011-06-30")
].copy()

# =========================
# RESAMPLE
# =========================
def robust_bin(series, rule="2h", min_count=5):
    med = series.resample(rule, origin="start_day").median()
    cnt = series.resample(rule, origin="start_day").count()
    med[cnt < min_count] = np.nan
    return med.dropna()

binned = robust_bin(june.set_index("start_time")["si"])

smooth = binned.rolling(window=10, center=True, min_periods=3).mean()

# =========================
# FIGURE 1 — TIME SERIES
# =========================
fig1, ax_ts = plt.subplots(figsize=(9.0, 4.8))

ax_ts.plot(binned.index, binned.values,
           linewidth=0.8, marker="o", markersize=2,
           color="orange", label="2-hr median SI", zorder=3)

ax_ts.plot(smooth.index, smooth.values,
           linewidth=2.0, alpha=0.6, color="sienna", zorder=2)

for t0, t1, label, color in CME_WINDOWS:
    ax_ts.axvspan(pd.Timestamp(t0), pd.Timestamp(t1),
                  alpha=0.12, color=color, zorder=1)

ax_ts.set_xlabel("Date")
ax_ts.set_ylabel("Scintillation Index (SI)")
ax_ts.legend(loc="upper left")
ax_ts.grid(True, linestyle="--", alpha=0.3)
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter("%d June"))
plt.suptitle("CME Detection - June 2011", fontsize=14)
plt.savefig("ips_timeseries_june.png", dpi=300)
plt.show()

# =========================
# SKY MAP FUNCTION
# =========================
def sky_map(ax, df_window, title, accent_color, vmin=None, vmax=None):
    ra_rad = np.deg2rad(df_window["ra"].values)
    r = np.deg2rad(90 - np.abs(df_window["dec"].values))
    si = df_window["si"].values

    if vmin is None: vmin = si.min()
    if vmax is None: vmax = np.percentile(si, 95)

    sizes = 10 + 250 * (si - vmin) / max(vmax - vmin, 1e-12)
    sizes = np.clip(sizes, 4, 300)

    sc = ax.scatter(ra_rad, r,
                    c=si, s=sizes,
                    cmap="YlOrRd", vmin=vmin, vmax=vmax,
                    alpha=0.75, linewidths=0.3,
                    edgecolors="0.3", zorder=3)

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_rlabel_position(135)
    ax.set_rticks([np.deg2rad(d) for d in [0, 30, 60, 90]])
    ax.set_yticklabels(["pole", "60°", "30°", "eq."], fontsize=7)
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 45)))
    ax.set_xticklabels([f"{h}h" for h in range(0, 24, 3)], fontsize=7)
    ax.grid(True, linestyle=":", alpha=0.3)
    ax.set_title(title, fontsize=9, pad=10, color=accent_color)

    return sc, vmin, vmax


In [ ]:
# =========================
# FIGURE 2 — MOLLWEIDE FULL-SKY MAP
# =========================
fig2, axes = plt.subplots(
    1, 2,
    subplot_kw={"projection": "mollweide"},
    figsize=(12, 3.5)
)
fig2.subplots_adjust(wspace=0.15)

def sky_map_mollweide(ax, df_window, title, accent_color, vmin=None, vmax=None):
    ra_deg  = df_window["ra"].values        # 0–360°
    dec_deg = df_window["dec"].values
    si      = df_window["si"].values

    # Wrap to [-180, +180] so RA=0h stays at center
    ra_shifted = ra_deg.copy()
    ra_shifted[ra_shifted > 180] -= 360
    ra_rad  = -np.deg2rad(ra_shifted)       # flip: east on left
    dec_rad =  np.deg2rad(dec_deg)

    if vmin is None: vmin = si.min()
    if vmax is None: vmax = np.percentile(si, 95)

    sizes = 15 + 200 * (si - vmin) / max(vmax - vmin, 1e-12)
    sizes = np.clip(sizes, 6, 280)

    sc = ax.scatter(ra_rad, dec_rad,
                    c=si, s=sizes,
                    cmap="YlOrRd", vmin=vmin, vmax=vmax,
                    alpha=0.8, linewidths=0.3, edgecolors="0.35",
                    zorder=3)

    # RA tick labels: 0h at center, increasing outward both sides
    ax.set_xticklabels([
        "14h","16h","18h","20h","22h",
        "0h",
        "2h","4h","6h","8h","10h"
    ], fontsize=7.5)
    ax.yaxis.set_tick_params(labelsize=7.5)

    # Celestial equator
    eq_ra = np.linspace(-np.pi, np.pi, 500)
    ax.plot(eq_ra, np.zeros_like(eq_ra),
            color="0.5", linewidth=0.6, linestyle="--", alpha=0.5)

    # Ecliptic — centered on RA=0h
    ecl_lon = np.linspace(0, 2 * np.pi, 500)
    ecl_lat = np.deg2rad(23.44) * np.sin(ecl_lon)
    ecl_ra  = ecl_lon.copy()
    ecl_ra[ecl_ra > np.pi] -= 2 * np.pi    # wrap to [-π, +π]
    # ax.plot(-ecl_ra, ecl_lat,
    #         color="gold", linewidth=0.9, linestyle="-", alpha=0.6)

    ax.set_title(title, fontsize=10, pad=12, color=accent_color, fontweight="normal")
    ax.grid(True, linestyle=":", alpha=0.25, linewidth=0.5)

    return sc

# Shared SI scale
vmin_g = df["si"].min()
vmax_g = np.percentile(df["si"].values, 95)

sky_panels = [
    (axes[0], "2011-06-06", "2011-06-07", "June 6–7",  "#000000"),
    (axes[1], "2011-06-18", "2011-06-21", "June 18–21", "#000000"),
]

for ax_s, t0, t1, title, color in sky_panels:
    mask = (df["start_time"] >= t0) & (df["start_time"] <= t1)
    sc = sky_map_mollweide(ax_s, df[mask], title, color, vmin_g, vmax_g)

# Annotation: Dec bands
for ax_s in axes:
    for dec_deg, label, va in [(+30, "+30°", "bottom"), (-30, "−30°", "top")]:
        ax_s.axhline(np.deg2rad(dec_deg),
                     color="0.6", linewidth=0.4, linestyle=":", alpha=0.5)
        ax_s.text(np.deg2rad(170), np.deg2rad(dec_deg) + np.deg2rad(2 if va=="bottom" else -4),
                  label, fontsize=6.5, color="0.5", ha="right")

# Colorbar
cbar = fig2.colorbar(sc, ax=axes.tolist(),
                     orientation="horizontal",
                     pad=0.08, fraction=0.05, aspect=50)
cbar.set_label("Scintillation Index (SI)", fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Legend for reference lines
axes[0].plot([], [], color="0.5",  linestyle="--", linewidth=0.8, label="Celestial equator")
# axes[0].plot([], [], color="gold", linestyle="-",  linewidth=0.9, label="Ecliptic")
axes[0].legend(loc="lower left", fontsize=7, framealpha=0.8)

fig2.suptitle("CME Sky Coverage - June 2011",
              fontsize=14, fontweight="normal", y=1.01)

plt.savefig("ips_skymaps_june.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import datetime as dt

df['date'] = df['start_time'].dt.date
counts = df.groupby('date')['si'].count()
counts.loc[dt.date(2011, 6, 7)]

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm
from scipy.sparse import lil_matrix, csr_matrix

from astropy.time import Time
from astropy.coordinates import get_body_barycentric_posvel
import astropy.units as u

# ── Grid (higher effective resolution by cutting at 1 AU) ──
GRID_R_AU     = np.linspace(0.1, 1.0, 40)   # denser radial grid
GRID_LON_DEG  = np.linspace(0, 360, 91)     # finer longitude
GRID_LAT_DEG  = np.linspace(-90, 90, 61)    # finer latitude

STEP_AU       = 0.04
N_SIRT_ITER   = 50
ELONG_MIN_DEG = 20

nr   = len(GRID_R_AU) - 1
nlon = len(GRID_LON_DEG) - 1
nlat = len(GRID_LAT_DEG) - 1

r_mid   = 0.5*(GRID_R_AU[:-1]    + GRID_R_AU[1:])
lon_mid = 0.5*(GRID_LON_DEG[:-1] + GRID_LON_DEG[1:])
lat_mid = 0.5*(GRID_LAT_DEG[:-1] + GRID_LAT_DEG[1:])

def vox(ir, il, ib):
    return ir*(nlon*nlat) + il*nlat + ib

# ── Coordinate transforms ──
def obliquity(t):
    T = (t.jd-2451545.0)/36525.0
    return np.radians((84381.406-46.836769*T)/3600.0)

def Rx(a):
    c,s = np.cos(a), np.sin(a)
    return np.array([[1,0,0],[0,c,-s],[0,s,c]])

earth_cache = {}
def earth_xyz(iso):
    key = iso[:10]
    if key not in earth_cache:
        t = Time(iso, format="isot", scale="utc")
        e = get_body_barycentric_posvel("earth", t)[0]
        s = get_body_barycentric_posvel("sun",   t)[0]
        v = np.array([(e.x-s.x).to(u.AU).value,
                      (e.y-s.y).to(u.AU).value,
                      (e.z-s.z).to(u.AU).value])
        earth_cache[key] = Rx(-obliquity(t)) @ v
    return earth_cache[key]

def los_unit(ra, dec, iso):
    t = Time(iso, format="isot", scale="utc")
    v = np.array([
        np.cos(np.radians(dec))*np.cos(np.radians(ra)),
        np.cos(np.radians(dec))*np.sin(np.radians(ra)),
        np.sin(np.radians(dec))
    ])
    return Rx(-obliquity(t)) @ v

# ── Load data ──
print("Loading data...")
with open("ips_continuous_results.json") as f:
    raw = json.load(f)

obs = [d for d in raw
       if d["type"]=="source"
       and d.get("elong_deg",0) > ELONG_MIN_DEG
       and d["si"] > 0
       and d.get("ra") is not None]

for d in obs:
    d["dt"] = datetime.fromisoformat(d["start_time"])

print(f"{len(obs)} observations")

# ── Precompute geometry ──
print("Earth positions...")
for d in tqdm(obs):
    d["exyz"] = earth_xyz(d["start_time"])

print("LOS vectors...")
for d in tqdm(obs):
    d["los"] = los_unit(d["ra"], d["dec"], d["start_time"])

# ── Build system matrix ──
print("Building system matrix...")
N  = len(obs)
NV = nr*nlon*nlat

A  = lil_matrix((N, NV), dtype=np.float32)
SI = np.array([d["si"] for d in obs])

for i,d in enumerate(tqdm(obs)):
    o = d["exyz"]
    v = d["los"]

    for t in np.arange(0, 1.2, STEP_AU):   # only up to ~1 AU
        p = o + t*v
        r = np.linalg.norm(p)

        if r < GRID_R_AU[0] or r >= GRID_R_AU[-1]:
            continue

        lon = np.degrees(np.arctan2(p[1], p[0])) % 360
        lat = np.degrees(np.arcsin(np.clip(p[2]/r, -1, 1)))

        if lat < GRID_LAT_DEG[0] or lat >= GRID_LAT_DEG[-1]:
            continue

        ir   = np.searchsorted(GRID_R_AU, r, "right") - 1
        ilon = np.searchsorted(GRID_LON_DEG, lon, "right") - 1
        ilat = np.searchsorted(GRID_LAT_DEG, lat, "right") - 1

        if 0<=ir<nr and 0<=ilon<nlon and 0<=ilat<nlat:
            A[i, vox(ir,ilon,ilat)] += STEP_AU/(r**2)

A = csr_matrix(A)

# ── Normalize ──
cs = np.array(A.sum(0)).flatten(); cs[cs==0]=1
rs = np.array(A.sum(1)).flatten(); rs[rs==0]=1

x = np.full(NV, np.median(SI)/np.mean(cs))

# ── SIRT reconstruction ──
print("Running SIRT...")
for _ in tqdm(range(N_SIRT_ITER)):
    res = SI - A.dot(x)
    x   = np.maximum(x + (1/cs)*(A.T.dot((1/rs)*res)), 0)

X3D = x.reshape(nr, nlon, nlat)

# ── Convert to Cartesian ──
print("Preparing 3D plot...")

R, LON, LAT = np.meshgrid(r_mid,
                          np.radians(lon_mid),
                          np.radians(lat_mid),
                          indexing="ij")

X = R * np.cos(LAT) * np.cos(LON)
Y = R * np.cos(LAT) * np.sin(LON)
Z = R * np.sin(LAT)

VAL = X3D

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.linewidth": 0.8,
    "axes.labelsize": 11,
    "axes.titlesize": 14,
    "axes.titlepad": 10,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "xtick.top": True,
    "ytick.right": True,
    "legend.fontsize": 9,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "0.7",
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})

from skimage.measure import marching_cubes
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.ndimage import map_coordinates
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator

print("Generating clean isosurface...")

V = X3D.copy()
V_norm = (V - V.min()) / (V.max() - V.min())
lvl = np.percentile(V_norm, 95)

verts, faces, _, _ = marching_cubes(V_norm, level=lvl)

def scale(val, vmin, vmax, n):
    return vmin + (val / (n - 1)) * (vmax - vmin)

xv = scale(verts[:, 0], r_mid.min(),   r_mid.max(),   len(r_mid))
yv = scale(verts[:, 1], lon_mid.min(), lon_mid.max(), len(lon_mid))
zv = scale(verts[:, 2], lat_mid.min(), lat_mid.max(), len(lat_mid))

r   = xv
lon = np.radians(yv)
lat = np.radians(zv)

Xc = r * np.cos(lat) * np.cos(lon)
Yc = r * np.cos(lat) * np.sin(lon)
Zc = r * np.sin(lat)

# ── Color by actual reconstructed SI from X3D ────────────────────────────
si_at_verts = map_coordinates(
    X3D,
    [verts[:, 0], verts[:, 1], verts[:, 2]],
    order=1,
    mode='nearest'
)
si_at_faces = si_at_verts[faces].mean(axis=1)

# avoid zeros for log
vmin = np.percentile(si_at_faces, 2)
vmax = np.percentile(si_at_faces, 98)
vmin = max(vmin, 1e-12)

norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
cmap = cm.magma
face_colors = cmap(norm(si_at_faces))

# ── Build and add surface ─────────────────────────────────────────────────
cart_verts = np.stack([Xc, Yc, Zc], axis=1)
triangles  = cart_verts[faces]

fig = plt.figure(figsize=(8, 8))
ax  = fig.add_subplot(111, projection='3d')

poly = Poly3DCollection(triangles,
                        facecolors=face_colors,
                        linewidth=0,
                        alpha=0.5)
ax.add_collection3d(poly)

# ── Colorbar ──────────────────────────────────────────────────────────────
from matplotlib.ticker import LogLocator, LogFormatter

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, shrink=0.45, pad=0.08, aspect=20)

cbar.set_label(r"Reconstructed $g$ (SIRT)", fontsize=10, labelpad=8)
cbar.ax.tick_params(labelsize=8)

# clean log ticks
cbar.locator = LogLocator(base=10)
cbar.formatter = LogFormatter(base=10)
cbar.update_ticks()
# ── Sun ───────────────────────────────────────────────────────────────────
ax.scatter(0, 0, 0, color="orange", s=120, zorder=5)

# ── Axes ──────────────────────────────────────────────────────────────────
ax.xaxis.set_major_locator(MaxNLocator(3))
ax.yaxis.set_major_locator(MaxNLocator(3))
ax.zaxis.set_major_locator(MaxNLocator(3))

ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
ax.set_xlabel("X (AU)", fontsize=9)
ax.set_ylabel("Y (AU)", fontsize=9)
ax.set_zlabel("Z (AU)", fontsize=9)

for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
    axis._axinfo["grid"]["color"] = (0.5, 0.5, 0.5, 0.08)
    axis._axinfo["tick"]["color"] = (0.2, 0.2, 0.2, 0.3)

ax.xaxis.pane.set_alpha(0.02)
ax.yaxis.pane.set_alpha(0.02)
ax.zaxis.pane.set_alpha(0.02)

# ── Wireframe sphere ──────────────────────────────────────────────────────
u = np.linspace(0, 2*np.pi, 80)
v = np.linspace(0,   np.pi, 40)
xs = np.outer(np.cos(u), np.sin(v))
ys = np.outer(np.sin(u), np.sin(v))
zs = np.outer(np.ones_like(u), np.cos(v))
ax.plot_wireframe(xs, ys, zs, color="gray", alpha=0.08, linewidth=0.5)

ax.set_box_aspect([1, 1, 1])
ax.set_title("IPS Tomography", fontweight="normal")
plt.tight_layout()
plt.savefig("ips_tomography.png", dpi=150)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.ticker import MaxNLocator, LogLocator, LogFormatterSciNotation

print("Generating volumetric tomography plot...")

# ── Select voxels ─────────────────────────────────────────────────────────
threshold = np.percentile(X3D, 95)   # show 90% of the volume
mask = X3D > threshold
ri, li, ai = np.where(mask)

r_v   = r_mid[ri]
lon_v = np.radians(lon_mid[li])
lat_v = np.radians(lat_mid[ai])

Xc = r_v * np.cos(lat_v) * np.cos(lon_v)
Yc = r_v * np.cos(lat_v) * np.sin(lon_v)
Zc = r_v * np.sin(lat_v)

si_vals = X3D[ri, li, ai]

vmin_c = np.percentile(si_vals, 1)
vmax_c = np.percentile(si_vals, 99)
vmin_c = max(vmin_c, 1e-12)
norm = mcolors.LogNorm(vmin=vmin_c, vmax=vmax_c)

# ── Sort far→near for depth ordering ─────────────────────────────────────
cam = np.array([2.5, 2.5, 1.5])
pts = np.stack([Xc, Yc, Zc], axis=1)
dist = np.linalg.norm(pts - cam, axis=1)
order = np.argsort(-dist)
Xc, Yc, Zc, si_vals = Xc[order], Yc[order], Zc[order], si_vals[order]

# ── Subsample if needed ───────────────────────────────────────────────────
max_pts = 500000
if len(Xc) > max_pts:
    idx = np.random.choice(len(Xc), max_pts, replace=False)
    idx.sort()
    Xc, Yc, Zc, si_vals = Xc[idx], Yc[idx], Zc[idx], si_vals[idx]

print(f"  Plotting {len(Xc)} voxels...")

# ── Plot ──────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(9, 8))
ax  = fig.add_subplot(111, projection='3d')

sc = ax.scatter(Xc, Yc, Zc,
                c=si_vals,
                cmap='magma',
                norm=norm,
                s=20,                  # big enough to see
                alpha=0.4,            # opaque enough to see
                depthshade=False,     # no fading
                edgecolors='none',
                rasterized=True)

# ── Colorbar ──────────────────────────────────────────────────────────────
cbar = fig.colorbar(sc, ax=ax, shrink=0.5, pad=0.08, aspect=20)
cbar.set_label(r"Reconstructed Scattering Strength $C_N^2$", fontsize=10, labelpad=8)
cbar.ax.tick_params(labelsize=8)
cbar.locator = LogLocator(base=10)
cbar.formatter = LogFormatterSciNotation(base=10)
cbar.update_ticks()

# ── Sun ───────────────────────────────────────────────────────────────────
ax.scatter(0, 0, 0, color="orange", s=150, zorder=5, edgecolors="darkgoldenrod", linewidths=0.5)

# ── Wireframe sphere ─────────────────────────────────────────────────────
u = np.linspace(0, 2*np.pi, 60)
v = np.linspace(0,   np.pi, 30)
r_outer = r_mid.max()
xs = r_outer * np.outer(np.cos(u), np.sin(v))
ys = r_outer * np.outer(np.sin(u), np.sin(v))
zs = r_outer * np.outer(np.ones_like(u), np.cos(v))
ax.plot_wireframe(xs, ys, zs, color="gray", alpha=0.2, linewidth=0.4)

# ── Axes ──────────────────────────────────────────────────────────────────
lim = r_mid.max() * 1.05
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
ax.set_xlabel("X (AU)", fontsize=9)
ax.set_ylabel("Y (AU)", fontsize=9)
ax.set_zlabel("Z (AU)", fontsize=9)
ax.xaxis.set_major_locator(MaxNLocator(3))
ax.yaxis.set_major_locator(MaxNLocator(3))
ax.zaxis.set_major_locator(MaxNLocator(3))

for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
    axis._axinfo["grid"]["color"] = (0.5, 0.5, 0.5, 0.2)
ax.xaxis.pane.set_alpha(0.02)
ax.yaxis.pane.set_alpha(0.02)
ax.zaxis.pane.set_alpha(0.02)

ax.set_box_aspect([1, 1, 1])
ax.view_init(elev=25, azim=135)
ax.set_title("IPS Tomography", fontweight="normal")

plt.tight_layout()
plt.savefig("ips_tomography_full_data.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.ticker import MaxNLocator, LogLocator, LogFormatterSciNotation

print("Generating volumetric tomography plot...")

# ── Select voxels ─────────────────────────────────────────────────────────
threshold = np.percentile(X3D, 80)
mask = X3D > threshold
ri, li, ai = np.where(mask)

r_v   = r_mid[ri]
lon_v = np.radians(lon_mid[li])
lat_v = np.radians(lat_mid[ai])

Xc = r_v * np.cos(lat_v) * np.cos(lon_v)
Yc = r_v * np.cos(lat_v) * np.sin(lon_v)
Zc = r_v * np.sin(lat_v)

si_vals = X3D[ri, li, ai]

# ── Clip half-space (REMOVE x > 0 side) ───────────────────────────────────
# clip_mask = ~((Xc < 0) & (Yc > 0) & (Zc < -0.5))
clip_mask = Xc > 0.5
# clip_mask = Zc < -0.5

Xc = Xc[clip_mask]
Yc = Yc[clip_mask]
Zc = Zc[clip_mask]
si_vals = si_vals[clip_mask]

# ── Color normalization ───────────────────────────────────────────────────
vmin_c = np.percentile(si_vals, 1)
vmax_c = np.percentile(si_vals, 99)
vmin_c = max(vmin_c, 1e-12)

norm = mcolors.LogNorm(vmin=vmin_c, vmax=vmax_c)

# ── Sort far→near for depth ordering ──────────────────────────────────────
cam = np.array([2.5, 2.5, 1.5])
pts = np.stack([Xc, Yc, Zc], axis=1)
dist = np.linalg.norm(pts - cam, axis=1)
order = np.argsort(-dist)

Xc = Xc[order]
Yc = Yc[order]
Zc = Zc[order]
si_vals = si_vals[order]

# ── Subsample if needed ───────────────────────────────────────────────────
max_pts = 500000
if len(Xc) > max_pts:
    idx = np.random.choice(len(Xc), max_pts, replace=False)
    idx.sort()
    Xc = Xc[idx]
    Yc = Yc[idx]
    Zc = Zc[idx]
    si_vals = si_vals[idx]

print(f"  Plotting {len(Xc)} voxels...")

# ── Plot ──────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(9, 8))
ax  = fig.add_subplot(111, projection='3d')

sc = ax.scatter(
    Xc, Yc, Zc,
    c=si_vals,
    cmap='magma',
    norm=norm,
    s=20,
    alpha=0.5,
    depthshade=False,
    edgecolors='none',
    rasterized=True
)

# ── Colorbar ──────────────────────────────────────────────────────────────
cbar = fig.colorbar(sc, ax=ax, shrink=0.5, pad=0.08, aspect=20)
cbar.set_label(r"Reconstructed $g$ (SIRT)", fontsize=10, labelpad=8)
cbar.ax.tick_params(labelsize=8)
cbar.locator = LogLocator(base=10)
cbar.formatter = LogFormatterSciNotation(base=10)
cbar.update_ticks()

# ── Sun ───────────────────────────────────────────────────────────────────
ax.scatter(
    0, 0, 0,
    color="orange",
    s=150,
    zorder=5,
    edgecolors="darkgoldenrod",
    linewidths=0.5,
    alpha=1.0
)

# ── Wireframe sphere ──────────────────────────────────────────────────────
u = np.linspace(0, 2*np.pi, 60)
v = np.linspace(0, np.pi, 30)

r_outer = r_mid.max()

xs = r_outer * np.outer(np.cos(u), np.sin(v))
ys = r_outer * np.outer(np.sin(u), np.sin(v))
zs = r_outer * np.outer(np.ones_like(u), np.cos(v))

ax.plot_wireframe(xs, ys, zs, color="gray", alpha=0.06, linewidth=0.4)

# ── Axes ──────────────────────────────────────────────────────────────────
lim = r_mid.max() * 1.05
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_zlim(-lim, lim)

ax.set_xlabel("X (AU)", fontsize=9)
ax.set_ylabel("Y (AU)", fontsize=9)
ax.set_zlabel("Z (AU)", fontsize=9)

ax.xaxis.set_major_locator(MaxNLocator(3))
ax.yaxis.set_major_locator(MaxNLocator(3))
ax.zaxis.set_major_locator(MaxNLocator(3))

for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
    axis._axinfo["grid"]["color"] = (0.5, 0.5, 0.5, 0.08)

ax.xaxis.pane.set_alpha(0.02)
ax.yaxis.pane.set_alpha(0.02)
ax.zaxis.pane.set_alpha(0.02)

ax.set_box_aspect([1, 1, 1])
ax.view_init(elev=25, azim=135)

ax.set_title("IPS Tomography (Clipped View)", fontweight="normal")

plt.tight_layout()
plt.savefig("ips_tomography_clipped.png", dpi=200)
plt.show()

In [ ]:
# ── Matplotlib style configuration ───────────────────────────────────────────
plt.rcParams.update({
    # Font
    "font.family":         "serif",
    "font.serif":          ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset":    "stix",

    # Axes
    "axes.linewidth":      0.8,
    "axes.labelsize":      15,
    "axes.titlesize":      15,
    "axes.titlepad":       15,
    "axes.spines.top":     True,
    "axes.spines.right":   True,

    # Ticks
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.major.size":    4,
    "ytick.major.size":    4,
    "xtick.minor.size":    2,
    "ytick.minor.size":    2,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.labelsize":     12,
    "ytick.labelsize":     12,
    "xtick.top":           True,
    "ytick.right":         True,

    # Legend
    "legend.fontsize":     9,
    "legend.framealpha":   0.9,
    "legend.edgecolor":    "0.7",

    # Figure
    "figure.dpi":          300,
    "savefig.dpi":         300,
    "savefig.bbox":        "tight",
    "savefig.pad_inches":  0.05,
})

# ─────────────────────────────────────────────
# FAST PARALLEL MONTHLY 3D TOMOGRAPHY
# ─────────────────────────────────────────────

from skimage.measure import marching_cubes
import matplotlib.pyplot as plt
from datetime import datetime
import numpy as np
from joblib import Parallel, delayed
from scipy.sparse import lil_matrix, csr_matrix

# ── NEW IMPORTS (moved to top) ──
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.ndimage import map_coordinates
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator

# ── GROUP BY MONTH ──
obs_all = []
for d in raw:
    if d.get("type") == "source":
        try:
            dt = datetime.fromisoformat(d["start_time"])
            obs_all.append({
                **d,
                "month": dt.strftime("%Y-%m")
            })
        except:
            continue

months = sorted(set(d["month"] for d in obs_all))[:8]

# ─────────────────────────────────────────────
# PRECOMPUTE GEOMETRY (VERY IMPORTANT)
# ─────────────────────────────────────────────
print("Precomputing geometry...")

for d in obs_all:
    if "exyz" not in d:
        d["exyz"] = earth_xyz(d["start_time"])
        d["los"]  = los_unit(d["ra"], d["dec"], d["start_time"])

# ─────────────────────────────────────────────
# CORE FUNCTION (PARALLEL UNIT)
# ─────────────────────────────────────────────
def process_month(month):

    obs = [d for d in obs_all
           if d["month"] == month
           and d.get("si",0) > 0
           and d.get("ra") is not None]

    if len(obs) < 50:
        return None

    N  = len(obs)
    NV = nr*nlon*nlat
    A  = lil_matrix((N,NV))
    SI = np.array([d["si"] for d in obs])

    for i,d in enumerate(obs):
        o,v = d["exyz"], d["los"]

        for t in np.arange(0,1.2,STEP_AU):
            p = o + t*v
            r = np.linalg.norm(p)

            if r < GRID_R_AU[0] or r >= GRID_R_AU[-1]:
                continue

            lon = np.degrees(np.arctan2(p[1],p[0])) % 360
            lat = np.degrees(np.arcsin(p[2]/r))

            ir   = np.searchsorted(GRID_R_AU, r) - 1
            ilon = np.searchsorted(GRID_LON_DEG, lon) - 1
            ilat = np.searchsorted(GRID_LAT_DEG, lat) - 1

            if 0<=ir<nr and 0<=ilon<nlon and 0<=ilat<nlat:
                A[i, vox(ir,ilon,ilat)] += STEP_AU/(r**2)

    A = csr_matrix(A)

    cs = np.array(A.sum(0)).flatten(); cs[cs==0]=1
    rs = np.array(A.sum(1)).flatten(); rs[rs==0]=1

    x = np.full(NV, np.median(SI)/np.mean(cs))

    for _ in range(30):
        res = SI - A.dot(x)
        x = np.maximum(x + (1/cs)*(A.T.dot((1/rs)*res)), 0)

    X3D = x.reshape(nr, nlon, nlat)

    V = (X3D - X3D.min()) / (X3D.max() - X3D.min())
    level = np.percentile(V, 95)

    try:
        verts, faces, _, _ = marching_cubes(V, level=level)
    except:
        return None

    # ── MODIFIED RETURN ──
    return verts, faces, len(obs), X3D


# ─────────────────────────────────────────────
# RUN PARALLEL
# ─────────────────────────────────────────────
print("Running parallel reconstruction...")

results = Parallel(n_jobs=-1)(
    delayed(process_month)(month) for month in months
)

# ─────────────────────────────────────────────
# PLOT RESULTS
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(22, 12))
fig.patch.set_facecolor("white")

axes = []
for i in range(8):
    ax = fig.add_subplot(2,4,i+1, projection='3d')
    axes.append(ax)

for idx, (month, result) in enumerate(zip(months, results)):

    ax = axes[idx]

    if result is None:
        ax.set_visible(False)
        continue

    # ── MODIFIED UNPACK ──
    verts, faces, nobs, X3D_month = result

    def scale(val, vmin, vmax, n):
        return vmin + (val/(n-1))*(vmax-vmin)

    xv = scale(verts[:,0], r_mid.min(), r_mid.max(), len(r_mid))
    yv = scale(verts[:,1], lon_mid.min(), lon_mid.max(), len(lon_mid))
    zv = scale(verts[:,2], lat_mid.min(), lat_mid.max(), len(lat_mid))

    r   = xv
    lon = np.radians(yv)
    lat = np.radians(zv)

    Xc = r * np.cos(lat) * np.cos(lon)
    Yc = r * np.cos(lat) * np.sin(lon)
    Zc = r * np.sin(lat)

    si_at_verts = map_coordinates(
        X3D_month,
        [verts[:, 0], verts[:, 1], verts[:, 2]],
        order=1,
        mode='nearest'
    )
    # ── THIS LINE WAS MISSING ──
    si_at_faces = si_at_verts[faces].mean(axis=1)

    norm_per_month = mcolors.Normalize(
        vmin=np.percentile(si_at_faces, 2),
        vmax=np.percentile(si_at_faces, 98)
    )
    face_colors = plt.cm.magma(norm_per_month(si_at_faces))

    cart_verts = np.stack([Xc, Yc, Zc], axis=1)
    triangles  = cart_verts[faces]

    poly = Poly3DCollection(
        triangles,
        linewidth=0,
        alpha=0.6
    )
    # ── KEY: set after construction, bypasses internal culling ──
    poly.set_facecolor(face_colors)
    poly.set_edgecolor("none")
    
    # ── KEY: disable Matplotlib's shading which overrides your colors ──
    poly.set_zsort("min")         # or "max" — prevents z-fighting artifacts
    
    ax.add_collection3d(poly)


    u = np.linspace(0, 2*np.pi, 40)
    v = np.linspace(0, np.pi, 20)

    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))

    ax.plot_wireframe(xs, ys, zs, color="gray", alpha=0.08, linewidth=0.5)

    ax.scatter(0,0,0, color="orange", s=40)

    ax.set_xlim(-0.75,0.75)
    ax.set_ylim(-0.75,0.75)
    ax.set_zlim(-0.75,0.75)

    ax.xaxis.set_major_locator(MaxNLocator(3))
    ax.yaxis.set_major_locator(MaxNLocator(3))
    ax.zaxis.set_major_locator(MaxNLocator(3))

    ax.set_box_aspect([1,1,1])
    ax.set_facecolor("white")

    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis._axinfo["grid"]["color"] = (0.5,0.5,0.5,0.04)

    ax.xaxis.pane.set_alpha(0.01)
    ax.yaxis.pane.set_alpha(0.01)
    ax.zaxis.pane.set_alpha(0.01)

    label = datetime.strptime(month,"%Y-%m").strftime("%b %Y")
    ax.set_title(f"{label}\n$N={nobs:,}$", pad=2)

# hide unused
for j in range(len(months),8):
    axes[j].set_visible(False)

# ── SHARED COLORBAR (FIXED) ──
from matplotlib.ticker import FormatStrFormatter

# Use same normalization range as your plotted data
all_vals = []
for r in results:
    if r is not None:
        _, faces, _, X3D = r
        vals = X3D.flatten()
        all_vals.append(vals)

all_vals = np.concatenate(all_vals)

norm = mcolors.Normalize(
    vmin=np.percentile(all_vals, 2),
    vmax=np.percentile(all_vals, 98)
)

sm = plt.cm.ScalarMappable(cmap="magma", norm=norm)
sm.set_array([])

# Create dedicated axis on the RIGHT
cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])

cbar = fig.colorbar(sm, cax=cax)

# Label + clean ticks
cbar.set_label(r"Reconstructed $g$ (SIRT)", fontsize=13)

cbar.ax.yaxis.set_major_locator(MaxNLocator(5))
cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

cbar.ax.tick_params(
    labelsize=11,
    width=0.8,
    length=4,
    direction='in'
)

fig.subplots_adjust(
    left=0.02, right=0.98,
    top=0.92, bottom=0.05,
    wspace=0.10,
    hspace=0.35
)

fig.suptitle("Monthly IPS Tomography", fontweight="normal", y=1.05, fontsize=25)

plt.savefig("Monthly_ips_tomography.png", dpi=150)
plt.show()
plt.close()

In [ ]:
# ── Matplotlib style configuration ───────────────────────────────────────────
plt.rcParams.update({
    "font.family":         "serif",
    "font.serif":          ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset":    "stix",
    "axes.linewidth":      0.8,
    "axes.labelsize":      15,
    "axes.titlesize":      15,
    "axes.titlepad":       15,
    "axes.spines.top":     True,
    "axes.spines.right":   True,
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.major.size":    4,
    "ytick.major.size":    4,
    "xtick.minor.size":    2,
    "ytick.minor.size":    2,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.labelsize":     12,
    "ytick.labelsize":     12,
    "xtick.top":           True,
    "ytick.right":         True,
    "legend.fontsize":     9,
    "legend.framealpha":   0.9,
    "legend.edgecolor":    "0.7",
    "figure.dpi":          300,
    "savefig.dpi":         300,
    "savefig.bbox":        "tight",
    "savefig.pad_inches":  0.05,
})

from datetime import datetime
import numpy as np
from joblib import Parallel, delayed
from scipy.sparse import lil_matrix, csr_matrix
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator, LogLocator, LogFormatterSciNotation

# ── GROUP BY MONTH ──
obs_all = []
for d in raw:
    if d.get("type") == "source":
        try:
            dt = datetime.fromisoformat(d["start_time"])
            obs_all.append({
                **d,
                "month": dt.strftime("%Y-%m")
            })
        except:
            continue

months = sorted(set(d["month"] for d in obs_all))[:8]

# ─────────────────────────────────────────────
# PRECOMPUTE GEOMETRY
# ─────────────────────────────────────────────
print("Precomputing geometry...")

for d in obs_all:
    if "exyz" not in d:
        d["exyz"] = earth_xyz(d["start_time"])
        d["los"]  = los_unit(d["ra"], d["dec"], d["start_time"])

# ─────────────────────────────────────────────
# CORE FUNCTION (NO MARCHING CUBES NEEDED)
# ─────────────────────────────────────────────
def process_month(month):

    obs = [d for d in obs_all
           if d["month"] == month
           and d.get("si",0) > 0
           and d.get("ra") is not None]

    if len(obs) < 50:
        return None

    N  = len(obs)
    NV = nr*nlon*nlat
    A  = lil_matrix((N,NV))
    SI = np.array([d["si"] for d in obs])

    for i,d in enumerate(obs):
        o,v = d["exyz"], d["los"]

        for t in np.arange(0,1.2,STEP_AU):
            p = o + t*v
            r = np.linalg.norm(p)

            if r < GRID_R_AU[0] or r >= GRID_R_AU[-1]:
                continue

            lon = np.degrees(np.arctan2(p[1],p[0])) % 360
            lat = np.degrees(np.arcsin(p[2]/r))

            ir   = np.searchsorted(GRID_R_AU, r) - 1
            ilon = np.searchsorted(GRID_LON_DEG, lon) - 1
            ilat = np.searchsorted(GRID_LAT_DEG, lat) - 1

            if 0<=ir<nr and 0<=ilon<nlon and 0<=ilat<nlat:
                A[i, vox(ir,ilon,ilat)] += STEP_AU/(r**2)

    A = csr_matrix(A)

    cs = np.array(A.sum(0)).flatten(); cs[cs==0]=1
    rs = np.array(A.sum(1)).flatten(); rs[rs==0]=1

    x = np.full(NV, np.median(SI)/np.mean(cs))

    for _ in range(30):
        res = SI - A.dot(x)
        x = np.maximum(x + (1/cs)*(A.T.dot((1/rs)*res)), 0)

    X3D = x.reshape(nr, nlon, nlat)

    return len(obs), X3D


# ─────────────────────────────────────────────
# RUN PARALLEL
# ─────────────────────────────────────────────
print("Running parallel reconstruction...")

results = Parallel(n_jobs=-1)(
    delayed(process_month)(month) for month in months
)

# ─────────────────────────────────────────────
# PLOT RESULTS — SCATTER VOXELS
# ─────────────────────────────────────────────
print("Plotting...")

fig = plt.figure(figsize=(24, 12))
fig.patch.set_facecolor("white")

axes = []
for i in range(8):
    ax = fig.add_subplot(2, 4, i+1, projection='3d')
    axes.append(ax)

for idx, (month, result) in enumerate(zip(months, results)):

    ax = axes[idx]

    if result is None:
        ax.set_visible(False)
        continue

    nobs, X3D_month = result

    # ── Select voxels above threshold (95th percentile, like original) ─
    threshold = np.percentile(X3D_month, 99)
    mask = X3D_month > threshold
    ri, li, ai = np.where(mask)

    r_v   = r_mid[ri]
    lon_v = np.radians(lon_mid[li])
    lat_v = np.radians(lat_mid[ai])

    Xc = r_v * np.cos(lat_v) * np.cos(lon_v)
    Yc = r_v * np.cos(lat_v) * np.sin(lon_v)
    Zc = r_v * np.sin(lat_v)

    si_vals = X3D_month[ri, li, ai]

    # ── LogNorm with 1st/99th percentile clipping (like original) ─────
    vmin_c = np.percentile(si_vals, 1)
    vmax_c = np.percentile(si_vals, 95)
    vmin_c = max(vmin_c, 1e-12)
    norm_m = mcolors.LogNorm(vmin=vmin_c, vmax=vmax_c)

    # ── Sort far→near for depth ordering ──────────────────────────────
    cam = np.array([2.5, 2.5, 1.5])
    pts = np.stack([Xc, Yc, Zc], axis=1)
    dist = np.linalg.norm(pts - cam, axis=1)
    order = np.argsort(-dist)
    Xc, Yc, Zc, si_vals = Xc[order], Yc[order], Zc[order], si_vals[order]

    # ── Subsample if too many ─────────────────────────────────────────
    max_pts = 500000
    if len(Xc) > max_pts:
        sel = np.random.choice(len(Xc), max_pts, replace=False)
        sel.sort()
        Xc, Yc, Zc, si_vals = Xc[sel], Yc[sel], Zc[sel], si_vals[sel]

    # ── Scatter plot (matched to original) ────────────────────────────
    sc = ax.scatter(Xc, Yc, Zc,
                    c=si_vals,
                    cmap='magma',
                    norm=norm_m,
                    s=20,
                    alpha=0.4,
                    depthshade=False,
                    edgecolors='none',
                    rasterized=True)

    # ── Colorbar with LogLocator (like original) ──────────────────────
    cb = fig.colorbar(sc, ax=ax, shrink=0.5, pad=0.08, aspect=20)
    cb.set_label(r"Reconstructed Scattering Strength $C_N^2$", fontsize=14, labelpad=8)
    cb.ax.tick_params(labelsize=12)
    cb.locator = LogLocator(base=10)
    cb.formatter = LogFormatterSciNotation(base=10)
    cb.update_ticks()

    # ── Sun (like original) ───────────────────────────────────────────
    ax.scatter(0, 0, 0, color="orange", s=150, zorder=5,
               edgecolors="darkgoldenrod", linewidths=0.5)

    # ── Wireframe sphere (like original) ──────────────────────────────
    u = np.linspace(0, 2*np.pi, 60)
    v = np.linspace(0,   np.pi, 30)
    r_outer = r_mid.max()
    xs = r_outer * np.outer(np.cos(u), np.sin(v))
    ys = r_outer * np.outer(np.sin(u), np.sin(v))
    zs = r_outer * np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(xs, ys, zs, color="gray", alpha=0.2, linewidth=0.4)

    # ── Axes (like original) ──────────────────────────────────────────
    lim = r_mid.max() * 1.05
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)

    ax.set_xlabel("X (AU)", fontsize=14)
    ax.set_ylabel("Y (AU)", fontsize=14)
    ax.set_zlabel("Z (AU)", fontsize=14)

    ax.xaxis.set_major_locator(MaxNLocator(3))
    ax.yaxis.set_major_locator(MaxNLocator(3))
    ax.zaxis.set_major_locator(MaxNLocator(3))

    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis._axinfo["grid"]["color"] = (0.5, 0.5, 0.5, 0.2)

    ax.xaxis.pane.set_alpha(0.02)
    ax.yaxis.pane.set_alpha(0.02)
    ax.zaxis.pane.set_alpha(0.02)

    ax.set_box_aspect([1, 1, 1])
    ax.set_facecolor("white")
    ax.view_init(elev=25, azim=135)

    label = datetime.strptime(month, "%Y-%m").strftime("%b %Y")
    ax.set_title(f"{label}\n$N={nobs:,}$", fontweight="normal", pad=2)

# ── Hide unused panels ───────────────────────────────────────────────────
for j in range(len(months), 8):
    axes[j].set_visible(False)

fig.subplots_adjust(
    left=0.02, right=0.98,
    top=0.90, bottom=0.05,
    wspace=0.20,
    hspace=0.35
)

fig.suptitle("IPS Tomography - Monthly", fontweight="normal", y=0.96, fontsize=25)

plt.savefig("Monthly_ips_tomography.png", dpi=300)
plt.show()
plt.close()
print("Done!")